# Perf metrics

Wall-time из `data/qa/command_timing.jsonl`:

1. **Последний запуск** каждой команды — сводный график.
2. **Все прогоны** — отдельный график по каждой команде.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from mobile.command_timing import DEFAULT_METRICS_PATH, load_command_metrics_df
from mobile.project_paths import PROJECT_ROOT

ROOT = PROJECT_ROOT
METRICS = ROOT / DEFAULT_METRICS_PATH

COMMAND_ORDER = (
    "build-stg-oktmo",
    "dq-stg-oktmo",
    "nb-stg-oktmo",
    "build-stg-time-zones",
    "dq-stg-time-zones",
    "nb-stg-time-zones",
    "build-stg-tac",
    "dq-stg-tac",
    "nb-stg-tac",
    "build-stg-oksm",
    "dq-stg-oksm",
    "nb-stg-oksm",
    "build-src-bs",
    "dq-src-bs",
    "nb-src-bs",
    "build-src-person",
    "dq-src-person",
    "nb-src-person",
    "build-src-excl",
    "dq-src-excl",
    "nb-src-excl",
    "build-src-mobile",
    "dq-src-mobile",
    "nb-src-mobile",
    "build-stg-event",
    "build-move-event",
    "dq-stg-event",
    "nb-stg-event",
    "build-stg-bs",
    "dq-stg-bs",
    "nb-stg-bs",
    "build-stg-geo-all",
    "dq-stg-geo-all",
    "nb-stg-geo-all",
    "nb-stg-msisdn-imei",
    "nb-stg-msisdn-imsi-operator",
    "nb-perf-metrics",
)


def _command_sort_key(command: str) -> tuple[int, int | str]:
    try:
        return (0, COMMAND_ORDER.index(command))
    except ValueError:
        return (1, command)

In [ ]:
raw = load_command_metrics_df(METRICS)
if raw.empty:
    print("Нет записей в", METRICS, "— выполните отдельные build-команды")
else:
    latest_idx = raw.groupby("command")["ts_utc"].idxmax()
    latest_per_command = raw.loc[latest_idx].copy()
    latest_per_command["_order"] = latest_per_command["command"].map(_command_sort_key)
    latest_per_command = latest_per_command.sort_values("_order").drop(columns="_order")

    cols = ["command", "ts_utc", "elapsed_total_sec", "status"]
    if "run_id" in latest_per_command.columns:
        cols.append("run_id")
    display(latest_per_command[cols])

    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(latest_per_command))))
    y = range(len(latest_per_command))
    values = latest_per_command["elapsed_total_sec"].astype(float)
    bars = ax.barh(y, values, color="#2a6fdb")
    ax.set_yticks(list(y))
    ax.set_yticklabels(latest_per_command["command"])
    ax.set_xlabel("elapsed_total_sec")
    ax.set_title("Последний запуск каждой команды")
    ax.grid(axis="x", alpha=0.3)
    xmax = float(values.max()) if len(values) else 1.0
    for bar, sec in zip(bars, values, strict=True):
        ax.text(
            bar.get_width() + xmax * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{sec:.1f}s",
            va="center",
            fontsize=9,
        )
    plt.tight_layout()
    plt.show()

In [ ]:
if raw.empty:
    pass
else:
    commands = sorted(raw["command"].dropna().unique(), key=_command_sort_key)
    n = len(commands)
    fig, axes = plt.subplots(n, 1, figsize=(12, max(3.2 * n, 4)), squeeze=False)

    for ax, command in zip(axes.ravel(), commands, strict=True):
        history = raw[raw["command"] == command].sort_values("ts_utc").reset_index(drop=True)
        x = range(len(history))
        values = history["elapsed_total_sec"].astype(float)
        bars = ax.bar(x, values, color="#27ae60", width=0.7)
        labels = history["ts_utc"].dt.strftime("%m-%d %H:%M")
        ax.set_xticks(list(x))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("sec")
        ax.set_title(f"{command} — все прогоны ({len(history)})")
        ax.grid(axis="y", alpha=0.3)
        ymax = float(values.max()) if len(values) else 1.0
        for bar, sec in zip(bars, values, strict=True):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ymax * 0.02,
                f"{sec:.1f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

    plt.tight_layout()
    plt.show()